In [3]:
for (year_to_process in 2018:2023) {
  library(data.table)
  library(readr)
  library(stringr)

  # Load necessary parameter files
  source("/home/resurreccion_cmc/drg-pipeline/data-cleaning/00a-parameters.r")
  suffix <- paste0(ifelse(to_sample, paste0(
    "_sampled_",
    sample_size_divisor, "_"
  ), "_full_"))

  # Define year to process

  # Validate suffix
  valid_suffix_pattern <- "_full_|_sampled_\\d+_"
  if (!grepl(valid_suffix_pattern, suffix)) {
    stop("Invalid suffix: ", suffix, ". Expected '_full_' or '_sampled_<sample_size_divisor>_'")
  }

  # Read and filter filenames
  files <- list.files(
    path = "~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/",
    pattern = paste0("map_", year_to_process, suffix, ".*fork_\\d+_part_\\d+\\.rds"),
    full.names = TRUE
  )

  if (length(files) == 0) {
    stop("No files found for the specified year and suffix: ", suffix)
  }

  # Function to parse file names
  parse_filename <- function(filename) {
    pattern <- paste0("map_(\\d{4})", suffix, "(\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}\\.\\d+)_fork_\\d+_part_\\d+\\.rds")
    matches <- str_match(filename, pattern)

    if (!is.na(matches[1])) {
      return(list(
        year = as.integer(matches[2]),
        timestamp = matches[3],
        filename = filename
      ))
    } else {
      return(NULL)
    }
  }

  parsed_files <- lapply(files, parse_filename)
  parsed_files <- Filter(Negate(is.null), parsed_files)

  # Select latest timestamp for the given year
  latest_timestamp <- max(sapply(parsed_files, `[[`, "timestamp"))
  latest_files <- Filter(function(x) x$timestamp == latest_timestamp, parsed_files)

  # Read and combine mappings
  final_icd_dt <- rbindlist(lapply(latest_files, function(file_info) {
    mappings_list <- readRDS(file_info$filename)
    return(mappings_list$icd_mappings)
  }), use.names = TRUE, fill = TRUE)

  final_rvs_dt <- rbindlist(lapply(latest_files, function(file_info) {
    mappings_list <- readRDS(file_info$filename)
    return(mappings_list$rvs_mappings)
  }), use.names = TRUE, fill = TRUE)

  # Deduplicate while allowing multiple mappings per source code
  final_icd_dt <- unique(final_icd_dt)
  final_rvs_dt <- unique(final_rvs_dt)

  # Create final list and save as .rds
  final_mappings_list <- list(final_icd_dt = final_icd_dt, final_rvs_dt = final_rvs_dt)
  output_rds <- paste0("~/drg-pipeline/data-cleaning/data/chkpts/final_mappings/final_map_", year_to_process, suffix, ".rds")
  saveRDS(final_mappings_list, output_rds)

  # Save debugging CSVs
  output_icd_csv <- paste0("~/drg-pipeline/data-cleaning/debug/icd_", year_to_process, suffix, ".csv")
  output_rvs_csv <- paste0("~/drg-pipeline/data-cleaning/debug/rvs_", year_to_process, suffix, ".csv")

  fwrite(final_icd_dt, output_icd_csv)
  fwrite(final_rvs_dt, output_rvs_csv)

  message("Final mappings saved to: ", output_rds)
  message("ICD mappings saved to: ", output_icd_csv)
  message("RVS mappings saved to: ", output_rvs_csv)
}


Parallelization: TRUE 


Final mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/final_mappings/final_map_2018_full_.rds

ICD mappings saved to: ~/drg-pipeline/data-cleaning/debug/icd_2018_full_.csv

RVS mappings saved to: ~/drg-pipeline/data-cleaning/debug/rvs_2018_full_.csv



Parallelization: TRUE 


Final mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/final_mappings/final_map_2019_full_.rds

ICD mappings saved to: ~/drg-pipeline/data-cleaning/debug/icd_2019_full_.csv

RVS mappings saved to: ~/drg-pipeline/data-cleaning/debug/rvs_2019_full_.csv



Parallelization: TRUE 


...

Final mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/final_mappings/final_map_2021_full_.rds

ICD mappings saved to: ~/drg-pipeline/data-cleaning/debug/icd_2021_full_.csv

RVS mappings saved to: ~/drg-pipeline/data-cleaning/debug/rvs_2021_full_.csv



Parallelization: TRUE 


Final mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/final_mappings/final_map_2022_full_.rds

ICD mappings saved to: ~/drg-pipeline/data-cleaning/debug/icd_2022_full_.csv

RVS mappings saved to: ~/drg-pipeline/data-cleaning/debug/rvs_2022_full_.csv



Parallelization: TRUE 


Final mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/final_mappings/final_map_2023_full_.rds

ICD mappings saved to: ~/drg-pipeline/data-cleaning/debug/icd_2023_full_.csv

RVS mappings saved to: ~/drg-pipeline/data-cleaning/debug/rvs_2023_full_.csv

